# Notebook 02: Zero-Shot Baseline Evaluation

### Research Question 1 (RQ1):
> *How well does the pretrained LLM perform financial sentiment classification without any task-specific fine-tuning?*

In this notebook, we evaluate the unmodified base model on our fixed test set using instruction prompting.

### What this notebook measures:
1. **Classification Quality:** Accuracy, Macro-F1, Weighted-F1, per-class Precision/Recall/F1, Confusion Matrix.
2. **Formatting Reliability:** Invalid output rate (how often the model fails to output one of the three sentiment classes).
3. **Inference Efficiency:** Average latency, median latency, 95th percentile latency, throughput (samples/sec), and peak GPU memory.
4. Saves predictions to `results/predictions/zero_shot.csv` and metrics to `results/metrics/zero_shot.json`.

In [1]:
# === MOUNT GOOGLE DRIVE & SETUP ===
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment')
sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'peft', 'accelerate', 'bitsandbytes',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'pyyaml'],
    check=True)

for d in ['results/metrics', 'results/predictions', 'results/figures',
          'results/logs', 'adapters/lora', 'adapters/qlora']:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print(f'src/ found: {os.path.exists("src")}')  # Must be True!
print('Setup complete! ✅')

Mounted at /content/drive
Working directory: /content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment
src/ found: True
Setup complete! ✅


In [2]:
# Step 2: Hardware check and Model Selection
import json
import torch


from src.model_utils import detect_hardware, print_hardware_summary, check_vram_feasibility

hw = detect_hardware()
print_hardware_summary(hw)

# Primary research model: Qwen2.5-3B-Instruct (3.09B parameters)
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

is_feasible, warning_msg = check_vram_feasibility(MODEL_NAME, method='zero_shot', hardware_info=hw)
print(warning_msg)
print(f'Active model: {MODEL_NAME}')

           HARDWARE & RUNTIME ENVIRONMENT SUMMARY           
CUDA Available:        True
GPU Name:              Tesla T4
GPU VRAM:              14.56 GB (14912.69 MB)
CUDA Version:          12.8
BF16 Supported:        True
PyTorch Version:       2.11.0+cu128
Transformers Version:  5.16.1
Compute Dtype:         bfloat16
Hardware check PASSED: 14.56 GB VRAM is sufficient for Qwen/Qwen2.5-3B-Instruct (ZERO_SHOT).
Active model: Qwen/Qwen2.5-3B-Instruct


In [3]:
# Step 3: Load the fixed test dataset
from src.data_utils import load_raw_dataset, create_stratified_split

df = load_raw_dataset('takala/financial_phrasebank', 'sentences_50agree')
_, _, test_df = create_stratified_split(df, 0.70, 0.15, 0.15, random_seed=42)
print(f'Test set loaded: {len(test_df)} samples.')

README.md:   0%|          | 0.00/9.58k [00:00<?, ?B/s]

financial_phrasebank.py:   0%|          | 0.00/6.04k [00:00<?, ?B/s]

Loading via `datasets` failed (Dataset scripts are no longer supported, but found financial_phrasebank.py). Using direct Hugging Face Hub download fallback...
Test set loaded: 727 samples.


In [4]:
# Step 4: Load Base Model and Tokenizer
from src.model_utils import load_tokenizer, load_base_model, count_parameters
from src.memory_utils import MemoryTracker, get_gpu_memory_stats

print(f'Loading tokenizer for {MODEL_NAME}...')
tokenizer = load_tokenizer(MODEL_NAME)

print(f'Loading base model {MODEL_NAME}...')
with MemoryTracker('Base Model Load') as load_tracker:
    model = load_base_model(MODEL_NAME, quantization=False)

params = count_parameters(model)
print(f'Total parameters: {params["total_parameters_millions"]} M')
print(load_tracker.summary())

Loading tokenizer for Qwen/Qwen2.5-3B-Instruct...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Total parameters: 3085.94 M
[Base Model Load] Peak Allocated: 5885.96 MB (5.748 GB) | Peak Reserved: 5958.0 MB (5.818 GB)


In [5]:
# Step 5: Run Zero-Shot Inference on Test Set
from src.data_utils import format_chat_prompt
from src.metrics import parse_model_output
import time

device = next(model.parameters()).device
raw_outputs = []
parsed_preds = []
latencies = []

print(f'Starting zero-shot inference on {len(test_df)} samples...')
t_start = time.perf_counter()

for idx, row in test_df.iterrows():
    prompt = format_chat_prompt(row['sentence'], tokenizer=tokenizer)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000.0)

    gen_text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    raw_outputs.append(gen_text)
    parsed_preds.append(parse_model_output(gen_text))

total_inference_time = time.perf_counter() - t_start
print(f'Inference completed in {total_inference_time:.1f} seconds.')

Starting zero-shot inference on 727 samples...
Inference completed in 455.7 seconds.


In [6]:
# Step 6: Compute Metrics and Save Results
from src.metrics import compute_classification_metrics, format_metrics_summary
from src.visualization import plot_confusion_matrix_heatmap
import numpy as np

y_true = test_df['label_text'].tolist()
metrics = compute_classification_metrics(y_true, parsed_preds)

# Add efficiency statistics
metrics['method'] = 'zero_shot'
metrics['model_name'] = MODEL_NAME
metrics['latency_avg_ms'] = round(float(np.mean(latencies)), 2)
metrics['latency_median_ms'] = round(float(np.median(latencies)), 2)
metrics['latency_p95_ms'] = round(float(np.percentile(latencies, 95)), 2)
metrics['throughput_samples_per_sec'] = round(len(test_df) / total_inference_time, 2)
metrics['peak_memory_gb'] = get_gpu_memory_stats()['peak_allocated_gb']

print(format_metrics_summary(metrics, 'Zero-Shot Baseline'))
print(f'Median Latency: {metrics["latency_median_ms"]} ms | P95: {metrics["latency_p95_ms"]} ms')

# Save metrics
os.makedirs('results/metrics', exist_ok=True)
os.makedirs('results/predictions', exist_ok=True)

with open('results/metrics/zero_shot.json', 'w') as f:
    json.dump(metrics, f, indent=2)

# Save predictions dataframe
pred_df = test_df.copy()
pred_df['raw_output'] = raw_outputs
pred_df['predicted_label'] = parsed_preds
pred_df['latency_ms'] = latencies
pred_df.to_csv('results/predictions/zero_shot.csv', index=False)

# Plot confusion matrix
plot_confusion_matrix_heatmap(metrics['confusion_matrix'], method_name='Zero-Shot Baseline')
print('Saved results to results/metrics/zero_shot.json and results/predictions/zero_shot.csv')

        CLASSIFICATION PERFORMANCE: ZERO-SHOT BASELINE
Accuracy:            0.7785 (77.85%)
Macro F1:            0.7733
Macro Precision:     0.7892
Macro Recall:        0.7820
Weighted F1:         0.7797
Invalid Output Rate: 2.20% (16/727)
-------------------------------------------------------
Per-Class Performance:
  Negative  ->  P: 0.7544 | R: 0.9451 | F1: 0.8390 (n=91)
  Neutral   ->  P: 0.8017 | R: 0.8519 | F1: 0.8260 (n=432)
  Positive  ->  P: 0.8116 | R: 0.5490 | F1: 0.6550 (n=204)
Median Latency: 663.61 ms | P95: 793.32 ms
Saved results to results/metrics/zero_shot.json and results/predictions/zero_shot.csv


### RQ1 Discussion

The zero-shot evaluation establishes the performance floor of the general instruction-tuned model.
Key aspects to observe:
1. Does the model recognize financial domain nuances, or does it default heavily to neutral?
2. What is the invalid output rate when prompted without task-specific tuning?
3. In the next notebook, we apply 16-bit LoRA to measure how much task adaptation improves these numbers.